In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import scipy.io as sio
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset
import os
from torch.amp import autocast, GradScaler

# Variable para controlar el uso de Google Drive o rutas locales
USE_DRIVE = False

# Configurar rutas según el valor de USE_DRIVE
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    DATASET = "kitpvc"
    DATA_PATH = f"/content/drive/MyDrive/UGR/deeplearning_ecgi/data/{DATASET}/"
    MODEL_PATH = f"/content/drive/MyDrive/UGR/deeplearning_ecgi/models/pytorch/{DATASET}/"
    PLOT_PATH = f"/content/drive/MyDrive/UGR/deeplearning_ecgi/plots/{DATASET}/"
else:
    # Establecer BASE_PATH como la ruta absoluta de la carpeta que contiene el script
    BASE_PATH = os.path.dirname(os.path.abspath(__file__))  # Esto obtiene la ruta de la carpeta del script
    DATASET = "kitpvc"
    DATA_PATH = os.path.join(BASE_PATH, f"data/{DATASET}/")
    MODEL_PATH = os.path.join(BASE_PATH, f"models/pytorch/{DATASET}/")
    PLOT_PATH = os.path.join(BASE_PATH, f"plots/{DATASET}/")

# Crear directorios si no existen
os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(PLOT_PATH, exist_ok=True)



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Usando dispositivo: {device}")

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True  # Optimización para convoluciones

# Funciones auxiliares
def tanh_normalization(y):
    return 2 * (y - np.min(y)) / (np.max(y) - np.min(y)) - 1

def load_data():
    data = sio.loadmat(os.path.join(DATA_PATH, f"{DATASET}_data.mat"))
    return data['ecg'], data['egm'], np.array(data['tags']), data['A']

# Cargar datos
X, y, label, A = load_data()

: 